In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import cv2

In [ ]:
import json

with open('data/label_encoding.json', 'r') as f:
    le = json.load(f)

In [ ]:
classes = list(le.keys())

In [ ]:
df = pd.read_csv('parse.csv')

def compile_prompt(row):
    return ' '.join([row['Brand'], row['Model'], row['Generation'], row['Years Active'], 'US-Canada', 'Photo'])

prompts = df.apply(compile_prompt, axis=1).values

In [ ]:
addon = list(set(['/'+x.split()[0].lower() for x in prompts]))

In [ ]:
class_orig = [x for j in addon for x in classes if x.startswith(j)]

In [ ]:
with open("class_orig.txt", "w") as file:
    for line in class_orig:
        file.write(line + "\n")

In [ ]:
with open("classes_tomatch.txt", "w", encoding='utf-8') as file:
    for line in prompts:
        file.write(line + "\n")

In [ ]:
import os
import json
import pandas as pd

with open("class_mapping_result.json", "r", encoding="utf-8") as f:
    class_map = json.load(f)

photos_dir = "data/images/photos"

data = []

for class_desc in os.listdir(photos_dir):
    folder_path = os.path.join(photos_dir, class_desc)

    if not os.path.isdir(folder_path):
        continue  

    class_label = class_map.get(class_desc)
    if not class_label:
        print(f"Warning: No mapping found for {class_desc}")
        continue

    for file in os.listdir(folder_path):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(folder_path, file)
            data.append({'img_path': image_path.replace('\\', '/')[18:], 'class':-1, 'class_name': class_label})

df_new = pd.DataFrame(data)

In [ ]:
df_new

In [ ]:
def read_image(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable at path: {path}")
    return (cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255).astype(np.float32)

In [ ]:
read_image('data/images/photos/Ram 2500 6th Generation 2025_Present US_Canada Photo/Ram25006thGeneration2025PresentUSCanadaPhoto7.png')

In [ ]:
train = pd.read_csv('data/train.txt')
val = pd.read_csv('data/val.txt')

In [ ]:
full_data = pd.concat([train, val]).reset_index(drop=True)

In [ ]:
full_data.to_csv('data/train_full.txt', index=False, sep=',')

In [ ]:
full_data['class'].nunique()

In [ ]:
full_data['class'].min()

In [ ]:
df = pd.read_csv('data/annotation_fixed.csv')

In [ ]:
df['class'].nunique()

In [ ]:
os.path.isfile("data/images"+df['img_path'].iloc[0])

In [ ]:
os.path.isfile("data/images/saab/93/1st/19224786.jpg")

In [ ]:
df['accessible'] = df['img_path'].apply(lambda x: os.path.isfile('data/images'+x))

In [ ]:
df['accessible'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

def save_annotations_from_cropped(csv_path, annotation_path):
    df = pd.read_csv(csv_path)

    # Добавляем фиктивные координаты (так как изображения уже обрезаны)
    df["x_1"] = 0
    df["y_1"] = 0
    df["x_2"] = 1
    df["y_2"] = 1

    # Стратифицированное разделение
    train, val = train_test_split(
        df,
        test_size=0.1,
        random_state=42,
        stratify=df["class"]
    )

    # Создание директории, если не существует
    if not os.path.isdir(annotation_path):
        os.makedirs(annotation_path)

    # Сохранение файлов
    train[["img_path", "x_1", "y_1", "x_2", "y_2", "class"]].to_csv(
        os.path.join(annotation_path, "train.txt"), index=False, sep=','
    )
    val[["img_path", "x_1", "y_1", "x_2", "y_2", "class"]].to_csv(
        os.path.join(annotation_path, "val.txt"), index=False, sep=','
    )

    print(f"✅ Сохранено: train ({len(train)}) и val ({len(val)}) в {annotation_path}")

In [ ]:
save_annotations_from_cropped("data/annotation_fixed_addon2.csv", "data/")

In [ ]:
df['class_name'] = df['img_path'].apply(lambda x: '/'.join(x.split('/')[:-1]))

In [ ]:
df = pd.concat([df, df_new])

In [ ]:
df = df[df['class'] != 227]

In [ ]:
df[df['class'] == 226]

In [ ]:
df['class'].value_counts()

In [ ]:
df['class'], uniques = pd.factorize(df['class_name'])

# Create mapping dictionary
label_to_class = dict(enumerate(uniques))

# Optionally, save to JSON
import json
with open("class_id_to_label.json", "w", encoding="utf-8") as f:
    json.dump(label_to_class, f, indent=2)

In [ ]:
df['class'].nunique()

In [ ]:
df

In [ ]:
def debug_random_crop(csv_path, image_root="./"):
    df = pd.read_csv(csv_path)

    # Выбираем случайную строку
    sample = df.sample(1).iloc[0]
    img_path = image_root + sample["img_path"]
    
    # Загружаем изображение
    img = cv2.imread(img_path)
    if img is None:
        print(f"❌ Не удалось загрузить изображение: {img_path}")
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Фиктивные координаты
    x1, y1, x2, y2 = sample['x_1'], sample['y_1'], sample['x_2'], sample['y_2'] #0, 0, 1, 1
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)  # прямоугольник

    # Показываем изображение
    plt.imshow(img)
    plt.title(f"{sample['img_path']} (class {sample['class']})")
    plt.axis("off")
    plt.show()

In [ ]:
debug_random_crop("data/val.csv", image_root="data/")

In [ ]:
img = cv2.imread('data/images/moskvitch_(azlk)/402/403_n_(19621965)/18938382.jpg')
if img is None:
    print(f"❌ Не удалось загрузить изображение: {img_path}")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Фиктивные координаты
x1, y1, x2, y2 = 0, 0, 1, 1
cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)  # прямоугольник

# Показываем изображение
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
contains_weird_chars('data/images/moskvitch_(azlk)/402/403¦н (1962-1965)/18938382.jpg')

In [ ]:
import re
from unidecode import unidecode

def safe_ascii_name(name):
    # Преобразуем в ASCII (удаляет большинство "странных" символов вроде ѝ, ¦ и т.п.)
    name = unidecode(name)
    
    # Удаляем символы: \ + - , 
    name = re.sub(r'[\\\+\,]', '', name)

    # Заменяем опасные символы (включая \n, \r, табуляцию) на подчёркивание
    name = re.sub(r'[<>:"/|?*\n\r\t]', '_', name)

    # Удалим всё, что НЕ латиница, цифра, подчеркивание, точка, пробел, скобки, дефис
    name = re.sub(r'[^\w\s\.\(\)\-]', '', name)

    # Заменяем подряд идущие пробелы/подчеркивания на одно подчёркивание
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'_+', '_', name)

    # Чистим начало/конец от мусора
    name = name.strip(' ._-')

    return name.lower()

In [ ]:
def contains_weird_chars(text):
    return bool(re.search(r'[^\x00-\x7Fа-яА-ЯёЁ0-9/\-_.() ]', text))

In [ ]:
from unidecode import unidecode
import re
import os
import json

# def safe_ascii_name(name):
#     name = unidecode(name)
#     name = re.sub(r'[\\\+\-\,]', '', name)
#     name = re.sub(r'[<>:"/|?*\n\r\t]', '_', name)
#     name = re.sub(r'\s+', '_', name)
#     name = re.sub(r'_+', '_', name)
#     name = name.strip(' ._-')
#     return name.lower()

def rename_tree_safe(root_path, save_map_path="rename_map.json", dry_run=False):
    rename_map = {}

    for dirpath, dirnames, filenames in os.walk(root_path, topdown=False):
        for filename in filenames:
            old_file_path = os.path.join(dirpath, filename)
            safe_filename = safe_ascii_name(filename)
            new_file_path = os.path.join(dirpath, safe_filename)

            # Абсолютный новый путь
            full_new_path = os.path.normpath(new_file_path)

            # Сохраняем ключ: filename → значение: полный путь от корня проекта
            file_key = filename
            file_value = full_new_path.replace("\\", "/")  # кроссплатформенно

            rename_map[file_key] = file_value

            if not dry_run and filename != safe_filename:
                try:
                    os.rename(old_file_path, new_file_path)
                    print(f"✅ File: {old_file_path} → {new_file_path}")
                except Exception as e:
                    print(f"❌ Ошибка с файлом {old_file_path}: {e}")
            elif dry_run:
                print(f"[DRY-RUN] File: {old_file_path} → {new_file_path}")

        # Переименовываем папки
        for dirname in dirnames:
            old_dir = os.path.join(dirpath, dirname)
            safe_dir = safe_ascii_name(dirname)
            if dirname != safe_dir:
                new_dir = os.path.join(dirpath, safe_dir)
                if not dry_run:
                    try:
                        os.rename(old_dir, new_dir)
                        print(f"✅ Dir: {old_dir} → {new_dir}")
                    except Exception as e:
                        print(f"❌ Ошибка с директорией {old_dir}: {e}")
                else:
                    print(f"[DRY-RUN] Dir: {old_dir} → {new_dir}")

    # Сохраняем JSON
    with open(save_map_path, "w", encoding="utf-8") as f:
        json.dump(rename_map, f, indent=2, ensure_ascii=False)

    print(f"\n📦 Сохранено в {save_map_path} — всего файлов: {len(rename_map)}")

In [ ]:
rename_tree_safe("data/images", save_map_path="rename_map_fix.json", dry_run=True)

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
df["filename"] = df["img_path"].apply(lambda x: x.split("/")[-1])

In [ ]:
with open('rename_map_fix.json', 'r', encoding="utf-8") as f:
    rename_map = json.load(f)

In [ ]:
rename_map['18938382.jpg']

In [ ]:
df['new_path'] = df['filename'].progress_apply(lambda x: rename_map[x].replace('data/images', ''))

In [ ]:
df['accessible'] = df['new_path'].progress_apply(lambda x: os.path.isfile('data/images'+x))
df['accessible'].value_counts()

In [ ]:
df

In [ ]:
df['class'] = df['class'].astype(int)
df[['img_path', 'class']].to_csv("data/annotation_fixed_addon2.csv", index=False)

In [ ]:
def normalize_annotation_paths(annotation_csv_path, output_csv_path=None):
    df = pd.read_csv(annotation_csv_path)

    def normalize_path(p):
        parts = p.split('/')
        parts = [safe_ascii_name(part) for part in parts]
        return '/'.join(parts)

    df["img_path"] = df["img_path"].astype(str).apply(normalize_path)

    if output_csv_path is None:
        output_csv_path = annotation_csv_path  # перезапишем

    df.to_csv(output_csv_path, index=False)
    print(f"✅ Обновлённые пути сохранены в {output_csv_path}")

In [ ]:
normalize_annotation_paths("data/annotation.csv", "data/annotation_clean.csv")

In [ ]:
df = pd.read_csv('data/annotation.csv')

In [ ]:
def normalize_path(p):
    parts = p.split('/')
    parts = [safe_ascii_name(part) for part in parts]
    return '/'.join(parts)

In [ ]:
df["img_path"].astype(str).apply(normalize_path)

In [ ]:
df['img_path'].apply(lambda x: x.split('/')[-1]).nunique()

In [ ]:
df['img_path'].nunique()

In [ ]:
def drop_duplicate_filenames(annotation_csv_path, output_path=None):
    df = pd.read_csv(annotation_csv_path)

    # Получаем только имя файла без пути
    df["filename"] = df["img_path"].apply(lambda x: x.split("/")[-1])

    # Оставляем только строки с уникальными именами файлов
    df = df.drop_duplicates(subset="filename")

    # Удаляем временную колонку
    df = df.drop(columns=["filename"])

    if output_path is None:
        output_path = annotation_csv_path  # перезаписать исходник

    df.to_csv(output_path, index=False)
    print(f"✅ Осталось {len(df)} уникальных имён файлов → сохранено в {output_path}")

drop_duplicate_filenames(
    annotation_csv_path="data/annotation.csv",
    output_path="data/annotation_unique.csv"
)